In [2]:
import pandas as pd
df = pd.read_csv("C:/Assessment2MLOps/data/synthetic_clinical_dataset.csv")
df.head(10)

,patient_id,age,sex,bmi,systolic_bp,diastolic_bp,glucose,cholesterol,creatinine,diabetes,hypertension,diagnosis,readmission_30d,mortality
0,1001,63,Male,24.1,131.0,81.0,136.2,210.3,1.04,0,0,Sepsis,0,0
1,1002,52,Male,20.8,118.0,91.0,136.5,207.9,1.49,0,1,Normal,0,0
2,1003,66,Male,30.0,144.0,73.0,100.0,179.5,1.15,0,0,Normal,0,0
3,1004,82,Male,16.8,112.0,76.0,117.3,228.3,0.59,0,0,Heart Failure,0,0
4,1005,50,Female,25.4,102.0,65.0,128.0,150.7,1.38,0,0,Normal,0,0
5,1006,50,Male,15.0,140.0,91.0,93.8,179.6,1.24,0,0,Pneumonia,0,0
6,1007,83,Male,19.0,118.0,84.0,72.4,216.2,1.30,0,1,Normal,0,0
7,1008,68,Female,21.7,119.0,93.0,122.2,220.1,1.34,0,0,Pneumonia,0,0
8,1009,46,Male,33.6,114.0,82.0,127.4,163.0,0.84,0,0,Heart Failure,0,0
9,1010,64,Female,31.1,116.0,93.0,123.7,265.0,1.32,0,1,Pneumonia,0,0


In [4]:
import yaml
cfg = yaml.safe_load(open("C:/Assessment2MLOps/config/config.yaml"))
print(cfg)
print(cfg["data"]["target"])

{'data': {'raw_path': 'data/synthetic_clinical_dataset.csv', 'target': 'readmission_30d', 'id_column': 'patient_id'}, 'schema': {'numeric': {'age': {'min': 18, 'max': 110}, 'bmi': {'min': 10, 'max': 70}, 'systolic_bp': {'min': 60, 'max': 250}, 'diastolic_bp': {'min': 30, 'max': 150}, 'glucose': {'min': 30, 'max': 600}, 'cholesterol': {'min': 50, 'max': 500}, 'creatinine': {'min': 0.1, 'max': 15}}, 'binary': ['diabetes', 'hypertension'], 'categorical': {'sex': ['Female', 'Male', 'Other'], 'diagnosis': ['Normal', 'Pneumonia', 'Heart Failure', 'Sepsis']}}}
readmission_30d


In [11]:
df["comorbidity_count"] = df["diabetes"] + df["hypertension"]

comorb_summary = df.groupby("comorbidity_count").agg(
    patients=("readmission_30d", "count"),
    readmitted=("readmission_30d", "sum"),
    readmission_rate=("readmission_30d", "mean"),
)
comorb_summary["readmission_rate_pct"] = (comorb_summary["readmission_rate"] * 100).round(1)
comorb_summary

,patients,readmitted,readmission_rate,readmission_rate_pct
comorbidity_count,,,,
0,6120,589,0.096242,9.6
1,3523,702,0.199262,19.9
2,357,99,0.277311,27.7


In [10]:
df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]

df["pp_band"] = pd.cut(
    df["pulse_pressure"],
    bins=[-1000, 40, 50, 60, 1000],          # band edges
    labels=["<40", "40-49", "50-59", "60+"],
    right=False,                             # 40 goes in "40-49", not "<40"
)
#Flag this in AI report, used AI to destermine the catagorisation of Pulse pressure Sys+diff as  pp_band
df.groupby("pp_band", observed=True).agg(
    patients=("readmission_30d", "count"),
    readmitted=("readmission_30d", "sum"),
    readmission_rate_mean=("readmission_30d", "mean"),
)

,patients,readmitted,readmission_rate_mean
pp_band,,,
<40,4102,565,0.137738
40-49,1852,270,0.145788
50-59,1638,211,0.128816
60+,2408,344,0.142857


In [12]:
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 35, 40, 1000],
    labels=["Underweight", "Normal", "Overweight", "Obese I", "Obese II", "Obese III"],
    right=False,
)

bmi_summary = df.groupby("bmi_category", observed=True).agg(
    patients=("readmission_30d", "count"),
    readmitted=("readmission_30d", "sum"),
    readmission_rate=("readmission_30d", "mean"),
)
bmi_summary["readmission_rate_pct"] = (bmi_summary["readmission_rate"] * 100).round(1)
bmi_summary

,patients,readmitted,readmission_rate,readmission_rate_pct
bmi_category,,,,
Underweight,727,104,0.143054,14.3
Normal,2931,408,0.139202,13.9
Overweight,3233,448,0.138571,13.9
Obese I,2194,321,0.146308,14.6
Obese II,779,95,0.121951,12.2
Obese III,136,14,0.102941,10.3


In [13]:
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 18, 40, 65, 80, 200],
    labels=["Under 18", "18-39", "40-64", "65-79", "80+"],
    right=False,
)

age_summary = df.groupby("age_band", observed=True).agg(
    patients=("readmission_30d", "count"),
    readmitted=("readmission_30d", "sum"),
    readmission_rate=("readmission_30d", "mean"),
)
age_summary["readmission_rate_pct"] = (age_summary["readmission_rate"] * 100).round(1)
age_summary

,patients,readmitted,readmission_rate,readmission_rate_pct
age_band,,,,
Under 18,204,20,0.098039,9.8
18-39,1833,269,0.146754,14.7
40-64,5089,687,0.134997,13.5
65-79,2039,284,0.139284,13.9
80+,835,130,0.155689,15.6


In [14]:
df["high_glucose"] = (df["glucose"] >= 126).astype(int)

glucose_summary = df.groupby("high_glucose").agg(
    patients=("readmission_30d", "count"),
    readmitted=("readmission_30d", "sum"),
    readmission_rate=("readmission_30d", "mean"),
)
glucose_summary["readmission_rate_pct"] = (glucose_summary["readmission_rate"] * 100).round(1)
glucose_summary

,patients,readmitted,readmission_rate,readmission_rate_pct
high_glucose,,,,
0,7946,1104,0.138938,13.9
1,2054,286,0.139241,13.9


In [15]:
# this is essentially telling me that the only effective measure is comorbidity.. I think.. Testing against diagnosis.
df.groupby("diagnosis").agg(
    patients=("readmission_30d", "count"),
    readmission_rate=("readmission_30d", "mean"),
)

,patients,readmission_rate
diagnosis,,
Heart Failure,1763,0.144073
Normal,5590,0.143113
Pneumonia,2005,0.135661
Sepsis,642,0.099688
